# Clasificación de 3 escenas con ViT preentrenado

Usamos el ViT-B/16 preentrenado en ImageNet de `keras-hub`
(`vit_base_patch16_224_imagenet`: 12 bloques, dimensión 768, 12 cabezas y unos
86 M de parámetros). Frente al ViT desde cero mide cuánto aporta el
preentrenamiento; frente a DenseNet121, que también parte de ImageNet, compara
atención y convolución en igualdad de condiciones.

El ajuste es en dos fases, como con DenseNet121:

1. **Backbone congelado**: se entrena solo la cabeza (`Dropout` + `Dense(3)`)
   sobre el token de clase, con `Adam(1e-3)`.
2. **Fine-tuning**: descongelamos los 2 últimos bloques del encoder y la
   `LayerNormalization` final, con `Adam(1e-5)`.

**Preprocesado.** Las imágenes de 128×128 se redimensionan a 224×224 y se
escalan a [-1, 1] dentro del modelo. No usamos el `ViTImageConverter` de
`keras-hub` porque necesita `tensorflow-text`, que no existe para Python 3.14.

**Pesos.** La primera ejecución descarga el preset (~345 MB). Solo guardamos los
pesos que cambia el ajuste: la cabeza, los 2 últimos bloques y la
`LayerNormalization` final (~14 M de parámetros, `finetuned.weights.npz`,
~57 MB). Al cargar, el modelo se reconstruye desde el preset y se le aplican
esos pesos. Con `FORCE_RETRAIN = True` se reentrena.

Instalación: `pip install --no-deps keras-hub kagglehub kagglesdk regex tqdm`
(`--no-deps` evita que pip cambie la versión de `keras`).

## Configuración

In [1]:
import keras
import keras_hub
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

import common_3scenes as cs

FORCE_RETRAIN = False
PRESET = "vit_base_patch16_224_imagenet"
VIT_INPUT = 224
FINE_TUNE_BLOCKS = 2
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cs.SEED)
np.random.seed(cs.SEED)

I0000 00:00:1795050124.965541  124358 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795050125.044130  124358 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795050126.080150  124358 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Carga del dataset y split común

In [2]:
data = cs.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cs.one_hot(y_train_labels)
y_val = cs.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Distribución de test:", {name: int(np.sum(y_test_labels == name)) for name in cs.CLASSES})

Train: (610, 128, 128, 3) | Validación: (108, 128, 128, 3) | Test: (240, 128, 128, 3)
Distribución de test: {'coast': 90, 'forest': 82, 'highway': 68}


## Modelo, entrenamiento y evaluación

`Input(128, 128, 3)` → flip horizontal y rotación 0.05 → `Resizing(224, 224)` →
`Rescaling(1/127.5, offset=-1)` → ViT-B/16 → token de clase → `Dropout(0.2)` →
`Dense(3, softmax)`. Fase 1: hasta 8 épocas; fase 2: hasta 6. Batch de 32 y
`EarlyStopping` sobre `val_accuracy`.

In [3]:
def build_vit_pretrained():
    inputs = layers.Input(shape=(cs.IMG_SIZE, cs.IMG_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.Resizing(VIT_INPUT, VIT_INPUT, interpolation="bilinear")(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)  # [0, 255] -> [-1, 1], as the preset

    backbone = keras_hub.models.Backbone.from_preset(PRESET)
    backbone.trainable = False

    tokens = backbone(x, training=False)
    class_token = tokens[:, 0]
    x = layers.Dropout(0.2)(class_token)
    outputs = layers.Dense(len(cs.CLASSES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="vit_pretrained_3scenes")
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model


def unfreeze_last_blocks(model, n_blocks=FINE_TUNE_BLOCKS):
    backbone = next(layer for layer in model.layers if isinstance(layer, keras_hub.models.Backbone))
    backbone.trainable = True
    backbone.get_layer("vit_patching_and_embedding").trainable = False
    encoder = backbone.get_layer("vit_encoder")
    for block in encoder.encoder_layers[:-n_blocks]:
        block.trainable = False


def fit_vit_pretrained(model):
    phase1_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
    history_phase1, seconds_phase1 = cs.train_keras_model(
        model, X_train, y_train, X_val, y_val,
        epochs=8, batch_size=32, callbacks=phase1_callbacks,
    )

    unfreeze_last_blocks(model)
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )
    phase2_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ]
    history_phase2, seconds_phase2 = cs.train_keras_model(
        model, X_train, y_train, X_val, y_val,
        epochs=6, batch_size=32, callbacks=phase2_callbacks,
    )

    history = pd.concat(
        [pd.DataFrame(history_phase1.history), pd.DataFrame(history_phase2.history)],
        ignore_index=True,
    )
    history.index = np.arange(1, len(history) + 1)
    return model, history, seconds_phase1 + seconds_phase2


def evaluate_vit_pretrained(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="ViT preentrenado (ImageNet)",
        family="IA generativa (Transformer)",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history),
        hyperparameters={
            "base": PRESET,
            "input": VIT_INPUT,
            "phase1_lr": 1e-3,
            "phase2_lr": 1e-5,
            "batch_size": 32,
            "epochs_phase1": 8,
            "epochs_phase2": 6,
            "dropout": 0.2,
            "fine_tuning_blocks": FINE_TUNE_BLOCKS,
            "trained_from_scratch": False,
        },
        notes=(
            "ViT-B/16 pretrained on ImageNet (keras-hub) with two-phase fine-tuning: "
            "frozen backbone, then the last encoder blocks unfrozen."
        ),
    )
    cs.save_results(
        "vit_preentrenado", metrics, y_test_labels, predicted, probabilities,
        test_paths=test_paths, history=history,
    )
    save_finetuned_weights(model, cs.result_dir("vit_preentrenado") / cs.ARTIFACTS["finetuned_weights"])
    return metrics


def save_finetuned_weights(model, path):
    # The full model (~450 MB) exceeds GitHub's limit; the frozen layers are the
    # public preset, so only the trainable weights (~57 MB) need to be stored.
    np.savez(path, *[keras.ops.convert_to_numpy(weight) for weight in model.trainable_weights])


def load_finetuned_weights(model, path):
    with np.load(path) as saved:
        values = [saved[f"arr_{i}"] for i in range(len(saved.files))]
    if len(values) != len(model.trainable_weights):
        raise ValueError(f"{path} has {len(values)} tensors, the model expects {len(model.trainable_weights)}")
    for weight, value in zip(model.trainable_weights, values):
        weight.assign(value)  # raises on a shape mismatch


def load_vit_pretrained(folder):
    weights_path = cs.result_dir(folder) / cs.ARTIFACTS["finetuned_weights"]
    if not weights_path.exists():
        return None
    model = build_vit_pretrained()  # frozen layers come from the ImageNet preset
    unfreeze_last_blocks(model)  # same trainable layout as when the weights were saved
    load_finetuned_weights(model, weights_path)
    return model


model, history, metrics, retrained = cs.load_or_train(
    "vit_preentrenado",
    build_model=build_vit_pretrained,
    fit_model=fit_vit_pretrained,
    evaluate_model=evaluate_vit_pretrained,
    load_model=load_vit_pretrained,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics["accuracy"], 4))
print("Parámetros:", f"{metrics['parameters']:,}")

[train] Training model in resultados/vit_preentrenado


E0000 00:00:1795050128.100561  124358 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



Epoch 1/8


20/20 - 127s - 6s/step - accuracy: 0.7672 - loss: 0.5908 - val_accuracy: 0.9352 - val_loss: 0.1664 - learning_rate: 0.0010


Epoch 2/8


20/20 - 114s - 6s/step - accuracy: 0.9869 - loss: 0.0895 - val_accuracy: 0.9537 - val_loss: 0.0870 - learning_rate: 0.0010


... (21 outputs left out)

Epoch 4: early stopping


Restoring model weights from the end of the best epoch: 1.


Results saved to: resultados/vit_preentrenado


Accuracy en test: 0.975
Parámetros: 85,800,963


## Observaciones

* 97.50 % en test, 8.75 puntos más que el ViT desde cero (McNemar p = 0.0001),
  con solo 12 épocas.
* Frente a DenseNet121 (98.33 %) la diferencia es de −0.8 puntos y no es
  significativa (p = 0.6875).
* Es el modelo más pesado: 86 M de parámetros y 121 ms por imagen en CPU,
  frente a 7 M y 16 ms de DenseNet121.